# 11a | Model Evaluation

"Which metric would you use, and why?" comes up in almost every ML interview. This lesson first drills the metric calculations, then goes through evaluation-related oral questions.

About 50 minutes.

## 1. The confusion matrix

| | Predicted positive | Predicted negative |
|---|---|---|
| **Actually positive** | TP (true positive) | FN (false negative, a miss) |
| **Actually negative** | FP (false positive, a false alarm) | TN (true negative) |

| Metric | Formula | Meaning |
|---|---|---|
| Accuracy | (TP + TN) / total | Share classified correctly |
| Precision | TP / (TP + FP) | Of those flagged, how many are real |
| Recall = Sensitivity = TPR | TP / (TP + FN) | Of the real ones, how many were found |
| Specificity = TNR | TN / (TN + FP) | Of the negatives, how many were classified correctly |
| FPR | FP / (FP + TN) = 1 − Specificity | Share of negatives falsely flagged |
| F1 | 2·P·R / (P + R) | Harmonic mean of precision and recall |

Mnemonic: **the denominator of precision is "predicted positive"; the denominator of recall is "actually positive."**

### ✏️ Exercise 1 (write it yourself)

A spam filter processed 1,000 emails, of which 100 are actually spam. It flagged 150 as spam, and 80 of those really are spam.

Compute `tp`, `fp`, `fn`, `tn`, as well as `precision`, `recall`, `accuracy`, and `f1`.

In [ ]:
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
tp = 80
fp = 150 - 80          # flagged as spam but actually not
fn = 100 - 80          # spam but not flagged
tn = 1000 - tp - fp - fn
precision = tp / (tp + fp)
recall = tp / (tp + fn)
accuracy = (tp + tn) / 1000
f1 = 2 * precision * recall / (precision + recall)
print(tp, fp, fn, tn, precision, recall, accuracy, f1)
```

91% accuracy looks good, but precision is only 53%: nearly half the flagged emails are legitimate. And "flag nothing" already has 90% accuracy. That's why you can't rely on accuracy when classes are imbalanced.

</details>

### ✏️ Exercise 2 (write it yourself)

Without sklearn, write a function `precision_recall_f1(y_true, y_pred)`: it takes two NumPy arrays of 0s and 1s and returns `(precision, recall, f1)`. If a denominator is 0, return `0.0` for that metric.

In [ ]:
import numpy as np

def precision_recall_f1(y_true, y_pred):
    # Write your code below
    pass

<details>
<summary><b>Hint</b></summary>

Count the three numbers with Boolean operations: `tp = np.sum((y_true == 1) & (y_pred == 1))`, and similarly for `fp` and `fn`. `a / b if b > 0 else 0.0` is a one-line conditional expression.

</details>

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np

def precision_recall_f1(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return precision, recall, f1
```

</details>

## 2. Regression metrics

- **MSE / RMSE:** mean of squared errors (then square root). Penalizes large errors heavily and is sensitive to outliers; RMSE is in the same units as y;
- **MAE:** mean of absolute errors. More robust to outliers;
- **R²:** the proportional improvement over "always predict the mean";
- **MAPE:** mean absolute percentage error. Blows up when y is close to 0.

Minimizing MSE gives the conditional **mean**; minimizing MAE gives the conditional **median**.

In NumPy, use `np.abs(x)` for elementwise absolute value, `np.sqrt(x)` for square root, and `np.mean(x)` for the mean (02a).

### ✏️ Exercise 3 (write it yourself)

Write a function `rmse_mae(y, y_hat)` that returns `(rmse, mae)`.

In [ ]:
import numpy as np

def rmse_mae(y, y_hat):
    # Write your code below
    pass

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np

def rmse_mae(y, y_hat):
    err = y - y_hat
    return np.sqrt(np.mean(err ** 2)), np.mean(np.abs(err))
```

In the first test only one prediction is off, by 4: MAE is 1 and RMSE is 2. A single large error affects RMSE much more.

</details>

## 3. Oral questions

**🗣 Say it out loud:** Should a cancer screening model focus more on sensitivity or specificity?

<details>
<summary><b>Reference answer</b></summary>

The main purpose of screening is not to miss patients; the cost of a missed diagnosis (delayed treatment) is usually far higher than that of a false alarm (one extra test), so prioritize **sensitivity (recall)**. But you can't look at only one metric: when prevalence is low, even with high specificity most positive results are still false positives (the Bayes problem in 05a), causing many unnecessary follow-up tests, costs, and anxiety. In practice, screening uses a high-sensitivity test, and those who test positive get a high-specificity confirmatory test. The threshold is set by the costs of both error types.

**Short version:** Sensitivity, because missing a case is usually far costlier than a false alarm. But with low prevalence even a specific test yields mostly false positives, so screening typically uses a sensitive test followed by a specific confirmatory one. The threshold should reflect both error costs.

</details>

**🗣 Say it out loud:** What does AUC = 0.8 mean? What are the limitations of AUC?

<details>
<summary><b>Reference answer</b></summary>

**Meaning:** pick one positive and one negative at random; the probability that the model scores the positive higher is 80%. 0.5 is random guessing and 1.0 is perfect ranking. It measures **ranking ability** and does not depend on a threshold.

**Limitations:** ① it doesn't reflect whether probabilities are calibrated; ② it looks overly optimistic under extreme class imbalance — also look at PR-AUC; ③ it averages over the whole curve, while the business may care about only one region (e.g., you can only act on the top 1% of scores — then look at precision@k or recall@k); ④ it doesn't reflect the costs of different errors.

**Short version:** It's the probability that a randomly chosen positive is scored above a randomly chosen negative, so it measures ranking quality independent of threshold. It says nothing about calibration, can look optimistic under heavy imbalance, and averages over operating points the business may not care about.

</details>

**🗣 Say it out loud:** What is calibration? Does a high AUC imply good calibration? When does calibration matter?

<details>
<summary><b>Reference answer</b></summary>

Good calibration means that among samples where the model says "probability 0.8," about 80% really are positive. Check it with a reliability curve (bin the predicted probabilities and plot each bin's mean prediction against its actual positive rate) or the Brier score.

High AUC does **not** imply good calibration: AUC only depends on ranking. Divide all predicted probabilities by 2 and AUC is unchanged, but calibration is completely wrong.

**When it matters:** when the probabilities themselves are used for decisions or calculations — e.g., expected revenue = probability × amount, risk-based pricing, communicating medical risk, or feeding probabilities into a downstream system.

**Fixes:** Platt scaling (fit a logistic regression on the scores) or isotonic regression, done on a separate validation set. Note that resampling and class weighting break calibration.

**Short version:** A model is calibrated if, among cases predicted at 80%, about 80% are positive. High AUC doesn't imply it, since AUC only depends on ranking. Calibration matters when probabilities feed decisions like expected value or pricing, and can be fixed with Platt scaling or isotonic regression on held-out data.

</details>

**🗣 Say it out loud:** When the data have timestamps, how should you split training and test sets?

<details>
<summary><b>Reference answer</b></summary>

Split by time: train on earlier data and test on later data, mimicking the real use of "predicting the future from the past." A random split lets the model see future information during training (temporal leakage) and overestimates performance because it never experiences real distribution change. Cross-validation should also use forward-rolling splits (TimeSeriesSplit). When building features, likewise use only data from before the prediction time.

**Short version:** Split by time: train on earlier data and test on later data, mirroring deployment. Random splits leak future information and hide distribution drift. For cross-validation, use forward-chaining splits.

</details>

**🗣 Say it out loud:** What is distribution shift? What kinds are there? How do you deal with it?

<details>
<summary><b>Reference answer</b></summary>

The data seen in production have a different distribution from the training data. ① **Covariate shift:** the feature distribution P(X) changes but the relationship between X and Y doesn't, e.g., the user population changes; ② **label shift:** P(Y) changes, e.g., the fraud rate rises; ③ **concept drift:** the relationship P(Y|X) itself changes, e.g., user behavior patterns change because of a pandemic.

**Responses:** monitor feature distributions, the distribution of predicted scores, and actual performance (when labels are available); retrain on a schedule or when triggered; use time-based validation to estimate the impact of drift; set alerts on important features.

**Short version:** The data in production differ from training. Covariate shift changes the feature distribution, label shift changes the class balance, and concept drift changes the relationship between features and target. I'd monitor input and score distributions and realized performance, and retrain on a schedule or trigger.

</details>

**🗣 Say it out loud:** A model has a high offline AUC but performs poorly after launch. What would you check?

<details>
<summary><b>Reference answer</b></summary>

In the order "engineering and data first, model second":

1. **Online/offline feature consistency:** feature computation logic, data sources, freshness (training used backfilled data, while online only real-time data is available);
2. **Data leakage:** offline used information not available online;
3. **Serving bugs:** missing features, default values, version mismatches, preprocessing out of sync;
4. **Distribution shift:** the live population differs from the training population, or time has moved on;
5. **Metric mismatch:** offline optimized AUC, while online looks at conversion or revenue, and the two don't necessarily agree;
6. **Label definition / delay:** online labels haven't fully come in yet;
7. **Feedback loops:** the model's output changed user behavior or subsequent data;
8. **Threshold and calibration:** whether they still fit the new distribution.

How to verify: replay real logged production requests offline and check whether online and offline predictions agree.

**Short version:** I'd check engineering and data before the model: training-serving feature skew, leakage, serving bugs, distribution shift in the live population, mismatch between the offline metric and the online goal, label delay, and feedback loops. Replaying logged production requests offline is a good way to isolate the cause.

</details>

**🗣 Say it out loud:** Why can't you use the test set to tune hyperparameters?

<details>
<summary><b>Reference answer</b></summary>

Hyperparameters are chosen by "seeing which value performs best on some data," and that selection process itself fits the randomness of that data. If you select on the test set, the test score is "cherry-picked" upward and is no longer an unbiased estimate of generalization performance. So choose hyperparameters with a validation set or cross-validation, and use the test set only once at the end. If you need an unbiased performance estimate while also selecting the model, use nested cross-validation.

**Short version:** Choosing hyperparameters by test performance fits the noise in the test set, so the test score becomes optimistically biased. Tune with a validation set or cross-validation and touch the test set once at the end; nested CV gives an unbiased estimate when you need both.

</details>

## Checkpoint

You can choose a metric for a business scenario and justify it; compute precision / recall / F1 by hand; and explain what AUC means, calibration, time-based splitting, distribution shift, and how to debug online/offline mismatches.

Next: **11b**.